# 07e · GBM, 블랙-숄즈, 파인만-카츠 (Geometric Brownian motion, Black–Scholes pricing and the Feynman–Kac link)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Shreve II §4.5 (Black-Scholes-Merton 방정식), §5.2–5.3 (위험중립 측도·가격결정 공식), §6.4 (Feynman-Kac 정리 6.4.1) — 주 교재 · Øksendal §8.2 (Feynman-Kac, Thm 8.2.1), §12.3 (option pricing) · Ross 12e §10.4 (무차익 논증의 가장 쉬운 서술) · Durrett 3e 6장 (이항모형 → BS 극한; 주의: Durrett 6장은 BM 이 아니라 금융) |
| 선수 노트북 | 07b (Itô 공식·Dynkin), 07c (EM 의 약수렴 차수), 07d (선형 SDE 의 적분인자), 06e (GBM 의 정의와 로그정규 분포) |
| 예상 소요 | 110분 |
| 상태 | draft |

06e 에서 기하 브라운 운동(geometric Brownian motion, GBM) $S_t=S_0\exp((\mu-\sigma^2/2)t+\sigma B_t)$ 를 정의했고, 07b 에서 그것이 $dS=\mu S\,dt+\sigma S\,dB$ 의 해임을 Itô 공식으로 확인했다. 이 노트북은 GBM 위에 세워진 가장 유명한 응용을 다룬다: **유럽형 옵션의 블랙-숄즈(Black–Scholes) 가격**. 핵심은 세 가지다. (1) 가격은 '평균 보수'가 아니라 드리프트를 $r$ 로 바꾼 가상의 세계(위험중립 측도)에서의 할인 기댓값이다. (2) 그 기댓값은 로그정규 적분을 두 조각으로 나누어 닫힌 형식 $S_0\Phi(d_1)-Ke^{-rT}\Phi(d_2)$ 으로 계산된다. (3) 같은 값은 편미분방정식(PDE)의 해이기도 하다 — 파인만-카츠(Feynman–Kac) 공식, 즉 07b 의 "Itô 로 드리프트를 없애면 마팅게일" 논법의 재등장이다. 몬테카를로(MC)와 PDE 라는 두 길을 한 화면에서 맞추어 보고, 실제 드리프트 $\mu$ 를 넣으면 왜 '가격'이 아닌지 숫자로 확인한다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import norm, lognorm
from scipy.optimize import brentq
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.brownian import gbm_paths
from spkit.sde import black_scholes_call, black_scholes_put, mc_option_price
from spkit.plots import plot_paths, plot_hist_vs_pdf

SEED, rng = rng_for("07e")
setup_plots()
N_PATHS = scale(20_000)    # Step 1 의 GBM 경로 수 (FAST(SPKIT_FAST=1) 모드에서는 1/10)
N_MC = scale(200_000)      # 가격 MC 의 표본 수
N_GRID = scale(50_000)     # u(0,x) 곡선의 공통 난수 개수
S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.2, 1.0   # 이 노트북의 기준 파라미터
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_MC={N_MC}, N_GRID={N_GRID}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (07d) OU 과정의 정상분포는 무엇이고, 정상분산에 $2\theta$ 가 나오는 이유는?

<details><summary>정답</summary>

$N(\mu,\sigma^2/(2\theta))$. 해 $X_t=\mu+(X_0-\mu)e^{-\theta t}+\sigma\int_0^te^{-\theta(t-s)}dB_s$ 의 확률적분 항에 등거리성을 쓰면 $\sigma^2\int_0^\infty e^{-2\theta s}ds=\sigma^2/(2\theta)$.

</details>

**Q2.** (07d) 정확 이산화와 EM 의 AR(1) 계수는 각각? EM 이 발산하는 조건은?

<details><summary>정답</summary>

정확: $e^{-\theta\Delta t}$, EM: $1-\theta\Delta t$. $|1-\theta\Delta t|>1$, 즉 $\theta\Delta t>2$ 이면 EM 이 발산한다 ($\theta\Delta t=2$ 는 계수 $-1$ 의 진동 경계).

</details>

**Q3.** (07d) 선형 SDE $dX=\theta(\mu-X)dt+\sigma dB$ 를 푸는 적분인자는?

<details><summary>정답</summary>

$e^{\theta t}$: $d(e^{\theta t}X_t)=\theta\mu e^{\theta t}dt+\sigma e^{\theta t}dB_t$ 로 드리프트에서 $X$ 가 사라진다. 오늘은 같은 아이디어를 $e^{-rt}u(t,S_t)$ 에 쓴다 (할인인자가 적분인자 역할).

</details>

**Q4.** (07c) EM 의 약수렴 차수와, GBM 에서 EM 평균의 정확한 값은?

<details><summary>정답</summary>

약차수 1; $E[X_n^{EM}]=X_0(1+\mu\Delta t)^n$ (결정론적 오일러와 같다). 오늘은 EM 이 필요 없다: GBM 의 $S_T$ 는 로그정규로 **정확히** 뽑을 수 있다 (`gbm_paths`, `mc_option_price`).

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **GBM.** $dS_t=\mu S_tdt+\sigma S_tdB_t$ 의 해는 $S_t=S_0\exp\big((\mu-\sigma^2/2)t+\sigma B_t\big)$ (07b: $\log S$ 에 Itô 공식을 쓰면 $d\log S=(\mu-\sigma^2/2)dt+\sigma dB$). 따라서 $\log S_T\sim N\big(\log S_0+(\mu-\sigma^2/2)T,\ \sigma^2T\big)$ — 로그정규(log-normal) 분포.
- **유럽형 콜/풋 (European call/put).** 만기 $T$ 에 콜은 $(S_T-K)^+$, 풋은 $(K-S_T)^+$ 를 지급한다. $K$ 는 행사가(strike), $r$ 은 무위험 이자율(연속복리).
- **위험중립 가격결정 (risk-neutral pricing) — 이 노트북의 공리.** 무차익(no-arbitrage) 가격 $=E^*[e^{-rT}\,\text{payoff}]$. 여기서 $E^*$ 아래에서 $S$ 는 **드리프트 $r$** 의 GBM: $S_T=S_0\exp\big((r-\sigma^2/2)T+\sigma\sqrt T\,Z\big)$, $Z\sim N(0,1)$. 실제 드리프트 $\mu$ 는 어디에도 등장하지 않는다. (측도변환/Girsanov 는 Season 2; 이항모형에서의 극한 유도는 Durrett 3e 6장; 복제 논증은 Ross 12e §10.4.)
- **블랙-숄즈 공식.** $C=S_0\Phi(d_1)-Ke^{-rT}\Phi(d_2)$, $d_1=\dfrac{\log(S_0/K)+(r+\sigma^2/2)T}{\sigma\sqrt T}$, $d_2=d_1-\sigma\sqrt T$. 풋은 $P=Ke^{-rT}\Phi(-d_2)-S_0\Phi(-d_1)$; **풋-콜 패리티** $C-P=S_0-Ke^{-rT}$.
- **BS PDE / 파인만-카츠.** $u(t,x)=E^*[e^{-r(T-t)}f(S_T)\mid S_t=x]$ 는 $u_t+rxu_x+\tfrac12\sigma^2x^2u_{xx}-ru=0$, $u(T,x)=f(x)$ 를 만족한다. 잔여만기 $\tau=T-t$ 로 쓰면 $u_t=-u_\tau$ 이므로 $-u_\tau+rxu_x+\tfrac12\sigma^2x^2u_{xx}-ru=0$.
- **델타 (delta).** $\Delta=u_x(0,S_0)=\partial C/\partial S_0=\Phi(d_1)$. pathwise 추정량 $e^{-rT}\mathbf 1_{\{S_T>K\}}S_T/S_0$ (C4 의 Greeks 예고).

### 2.2 정리 1 — 블랙-숄즈 공식 (위험중립 기댓값의 직접 계산; Shreve II §5.2–5.3, Ross 12e §10.4)

$S_T=S_0e^{(r-\sigma^2/2)T+\sigma\sqrt T\,Z}$, $Z\sim N(0,1)$ 이면

$$E^*[e^{-rT}(S_T-K)^+]=S_0\Phi(d_1)-Ke^{-rT}\Phi(d_2).$$

조각별로 $E^*[e^{-rT}S_T\mathbf 1_{\{S_T>K\}}]=S_0\Phi(d_1)$, $E^*[e^{-rT}K\mathbf 1_{\{S_T>K\}}]=Ke^{-rT}\Phi(d_2)$, $P^*(S_T>K)=\Phi(d_2)$.

가정이 왜 필요한가:
- **로그정규 $S_T$ (상수 $\sigma$, 연속 경로)**: 적분이 정규분포의 꼬리 $\Phi(\cdot)$ 로 닫히는 이유. $\sigma$ 가 랜덤이면 로그정규의 **혼합**이 되어 공식이 깨진다 (E3, 변동성 스마일).
- **드리프트가 $r$**: 무차익(복제) 논증의 결과. 실제 드리프트 $\mu$ 를 쓰면 다른 숫자가 나온다 (Step 5).
- **상수 $r$, 배당·거래비용 없음**: 할인이 $e^{-rT}$ 하나로 끝나고 복제가 주식+현금만으로 되는 이유.

<details><summary>증명</summary>

$S_T>K\iff(r-\sigma^2/2)T+\sigma\sqrt T\,Z>\log(K/S_0)\iff Z>-d_2$, 여기서
$$-d_2=\frac{\log(K/S_0)-(r-\sigma^2/2)T}{\sigma\sqrt T}.$$
(확인: $d_2=\frac{\log(S_0/K)+(r-\sigma^2/2)T}{\sigma\sqrt T}=d_1-\sigma\sqrt T$.)

**둘째 조각.** $E^*[e^{-rT}K\mathbf 1_{\{S_T>K\}}]=Ke^{-rT}P(Z>-d_2)=Ke^{-rT}\Phi(d_2)$ (표준정규의 대칭성). 특히 $P^*(S_T>K)=\Phi(d_2)$.

**첫째 조각.** $\phi$ 를 표준정규 밀도라 하면
$$E^*[e^{-rT}S_T\mathbf 1_{\{S_T>K\}}]=e^{-rT}S_0\int_{-d_2}^\infty e^{(r-\sigma^2/2)T+\sigma\sqrt T\,z}\phi(z)\,dz
=S_0\int_{-d_2}^\infty\frac{1}{\sqrt{2\pi}}\exp\Big(-\frac{z^2}2+\sigma\sqrt T\,z-\frac{\sigma^2T}2\Big)dz .$$
지수를 완전제곱으로: $-\frac{z^2}2+\sigma\sqrt T\,z-\frac{\sigma^2T}2=-\frac{(z-\sigma\sqrt T)^2}2$. 따라서 피적분함수는 $\phi(z-\sigma\sqrt T)$ 이고
$$=S_0\int_{-d_2}^\infty\phi(z-\sigma\sqrt T)\,dz=S_0P(Z'>-d_2-\sigma\sqrt T)=S_0\Phi(d_2+\sigma\sqrt T)=S_0\Phi(d_1).$$
($e^{-rT}$ 와 $e^{rT}$ 가 상쇄되는 것이 "드리프트 $r$" 가정이 쓰이는 지점이다.)

**빼면** $(S_T-K)^+=S_T\mathbf 1_{\{S_T>K\}}-K\mathbf 1_{\{S_T>K\}}$ 이므로 $C=S_0\Phi(d_1)-Ke^{-rT}\Phi(d_2)$.

**풋.** $(S_T-K)^+-(K-S_T)^+=S_T-K$ 의 할인 기댓값은 $E^*[e^{-rT}S_T]-Ke^{-rT}=S_0-Ke^{-rT}$ ($E^*[S_T]=S_0e^{rT}$: 06e 의 $E[S_t]=S_0e^{\mu t}$ 에 $\mu=r$). 따라서 $P=C-S_0+Ke^{-rT}=Ke^{-rT}(1-\Phi(d_2))-S_0(1-\Phi(d_1))=Ke^{-rT}\Phi(-d_2)-S_0\Phi(-d_1)$. ∎

수치 ($S_0=K=100,r=0.05,\sigma=0.2,T=1$): $d_1=0.35,d_2=0.15$, $\Phi(d_1)=0.636831$, $\Phi(d_2)=0.559618$, $C=63.6831-53.2325=10.4506$, $P=5.5735$. (4절에서 코드로 다시 계산한다.)

</details>

### 2.3 정리 2 — 파인만-카츠: BS PDE 와 위험중립 기댓값의 동치 (Shreve II §6.4 Thm 6.4.1, Øksendal §8.2 Thm 8.2.1)

$u\in C^{1,2}$ 가 $u_t+rxu_x+\tfrac12\sigma^2x^2u_{xx}-ru=0$, $u(T,x)=f(x)$ 를 만족하고 적절한 성장조건을 가지면
$$u(t,x)=E^*[e^{-r(T-t)}f(S_T)\mid S_t=x].$$
역으로 이 기댓값은 (충분히 매끄러우면) PDE 를 만족한다.

가정이 왜 필요한가:
- **$u\in C^{1,2}$**: $e^{-rt}u(t,S_t)$ 에 (시간의존) Itô 공식을 적용하려면 $u_t,u_x,u_{xx}$ 가 있어야 한다.
- **성장조건** ($|u(t,x)|\le Ce^{c\log^2x}$ 등): 확률적분 항 $\int e^{-rs}\sigma S_su_x\,dB_s$ 가 국소 마팅게일이 아니라 **진짜** 마팅게일이 되어 기댓값이 0 이 되려면 (07b Dynkin 공식의 주의점과 같다).
- **$S$ 가 위험중립 GBM**: 생성원(generator) $\mathcal L=rx\partial_x+\tfrac12\sigma^2x^2\partial_{xx}$ 가 PDE 의 공간 연산자가 되는 이유. 할인 항 $-ru$ 는 $e^{-rt}$ 곱의 미분에서 나온다.

<details><summary>증명 (sketch)</summary>

$Y_t=e^{-rt}u(t,S_t)$ 에 곱셈규칙과 시간의존 Itô 공식을 쓰면 ($dS=rS\,dt+\sigma S\,dB$, $(dS)^2=\sigma^2S^2dt$)
$$dY_t=e^{-rt}\Big[\big(u_t+rS_tu_x+\tfrac12\sigma^2S_t^2u_{xx}-ru\big)dt+\sigma S_tu_x\,dB_t\Big].$$
PDE 에 의해 $dt$ 항이 0 이므로 $Y$ 는 (성장조건 아래) 마팅게일. 따라서 $Y_t=E^*[Y_T\mid\mathcal F_t]=E^*[e^{-rT}f(S_T)\mid\mathcal F_t]$, 즉 $u(t,S_t)=E^*[e^{-r(T-t)}f(S_T)\mid\mathcal F_t]$. 마르코프 성질로 조건은 $S_t=x$ 만 남아 $u(t,x)=E^*[e^{-r(T-t)}f(S_T)\mid S_t=x]$.

역방향(기댓값이 PDE 를 만족)은 마르코프 성질로 $e^{-rt}u(t,S_t)$ 가 마팅게일임을 먼저 보이고, 매끄러우면 Itô 전개의 $dt$ 항이 0 이어야 함을 이용한다 — Øksendal Thm 8.2.1, Shreve II Thm 6.4.1. 수치 검증: 블랙-숄즈 공식을 $u(\tau,x)$ 로 놓고 유한차분 잔차를 계산하면 $\approx10^{-7}$ (Step 4).

</details>

### 2.4 정리 3 — 델타 $u_x(0,S_0)=\Phi(d_1)$ 과 pathwise 추정량

$$\frac{\partial}{\partial S_0}E^*[e^{-rT}(S_T-K)^+]=E^*\Big[e^{-rT}\mathbf 1_{\{S_T>K\}}\frac{\partial S_T}{\partial S_0}\Big]=E^*\Big[e^{-rT}\mathbf 1_{\{S_T>K\}}\frac{S_T}{S_0}\Big]=\Phi(d_1).$$

가정이 왜 필요한가:
- **미분과 기댓값의 교환**: payoff 가 Lipschitz 이면 (콜은 그렇다) 지배수렴으로 교환 가능. 디지털 옵션 $\mathbf 1_{\{S_T>K\}}$ 는 아니다 → pathwise 미분이 a.s. 0 이라 likelihood-ratio 법이 필요 (E2, C4).
- **$S_T$ 가 $S_0$ 에 비례**: $S_T=S_0e^{\cdots}$ 이므로 $\partial S_T/\partial S_0=S_T/S_0$.

<details><summary>증명 (sketch)</summary>

마지막 등식은 정리 1 의 첫 조각 $E^*[e^{-rT}S_T\mathbf 1_{\{S_T>K\}}]=S_0\Phi(d_1)$ 을 $S_0$ 로 나눈 것이다. 공식을 직접 미분해도 같다: $C_{S_0}=\Phi(d_1)+S_0\phi(d_1)\,\partial_{S_0}d_1-Ke^{-rT}\phi(d_2)\,\partial_{S_0}d_2$ 인데 $\partial_{S_0}d_1=\partial_{S_0}d_2$ 이고 $S_0\phi(d_1)=Ke^{-rT}\phi(d_2)$ (정리 1 증명의 완전제곱과 같은 항등식) 이므로 뒤 두 항이 상쇄된다. 감마 $u_{xx}=\phi(d_1)/(S_0\sigma\sqrt T)$ 도 같은 계산에서 나온다.

</details>

### 2.5 직관

옵션 가격은 '평균 보수'가 아니라 **복제 비용**이다. 주식과 현금으로 만기 보수를 복제할 수 있으면 가격은 복제 포트폴리오의 오늘 값이고, 이 값은 주가가 평균적으로 얼마나 오를지($\mu$)와 무관하다 — $\mu$ 에 대한 견해가 다른 두 사람도 같은 헤지를 하기 때문이다. 수학적으로 이것은 "드리프트를 $r$ 로 바꾼 가상의 세계에서 할인 기댓값을 취하라"로 정리된다 (위험중립). 블랙-숄즈 공식은 그 기댓값을 로그정규 적분으로 닫은 것이고, $\Phi(d_2)$ 는 위험중립 세계에서 옵션이 행사될 확률, $S_0\Phi(d_1)$ 은 행사될 때 받는 주식의 현재가치다. 파인만-카츠는 같은 값을 PDE 의 해로 보는 관점: 06d/07b 의 "Itô 로 드리프트를 없애면 마팅게일" 논법이 그대로 쓰인다. MC 와 PDE 는 같은 숫자로 가는 두 길이며, 이 노트북은 둘을 한 화면에서 맞춘다.

### 2.6 함정 (traps)

- 실제 드리프트 $\mu$ 를 넣고 할인 기댓값을 계산하면 **가격이 아니다** ($\mu=0.15$: 18.08 vs 10.45, Step 5).
- $\Phi(d_2)$ 는 '실제' 행사 확률이 아니라 **위험중립** 확률 (실제 $\mu=0.15$ 에서는 $\Phi(d_2')=0.742$ vs 0.560).
- MC 가격의 SE 는 할인 payoff 의 표준편차 ($\approx14.7$) 로 결정되어 200 000 경로에도 $\approx0.033$ — 공식이 있으면 공식이 낫고, 공식이 없을 때 (아시안·배리어, C4) 분산감소가 핵심이 된다.
- $u(t,x)$ 의 PDE 검증은 $t$ 가 아니라 잔여만기 $\tau=T-t$ 로 미분한다: $u_t=-u_\tau$ 부호를 틀리기 쉽다.
- 델타의 pathwise 추정은 payoff 가 Lipschitz 일 때만; 디지털 옵션의 미분은 델타함수라 pathwise 가 0 을 준다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — '옵션 가격에 왜 $\mu$ 가 안 들어가는가'를 복제 논증으로 3문장, 그리고 '파인만-카츠가 07b 의 Dynkin 공식과 어떻게 같은 아이디어인지'를 1문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

정리 1 의 식에 숫자를 넣으면 계산기와 $\Phi$ 표로 구할 수 있습니다 ($\Phi(0.35)=0.6368$, $\Phi(0.15)=0.5596$, $\Phi(0.65)=0.7422$, $\Phi(0.85)=0.8023$). 시뮬레이션 뒤 5절의 표에서 자기 예측과 비교합니다.

In [ ]:
predictions = {
    # S0=100, K=100, r=0.05, σ=0.2, T=1 인 유럽형 콜의 블랙-숄즈 가격은?
    "bs_call": None,
    # 같은 조건의 풋 가격은? (풋-콜 패리티 C - P = S0 - K e^{-rT} 를 쓰면 빠르다)
    "bs_put": None,
    # 위험중립 세계에서 만기에 내가격이 될 확률 P*(S_T > K) 는?
    "p_itm": None,
    # 콜의 델타 ∂C/∂S0 는?
    "delta": None,
    # 위험중립 기댓값 E*[S_T] 는?
    "E_ST": None,
    # 드리프트를 '실제' μ = 0.15 로 두고 E[e^{-rT}(S_T-K)^+] 를 계산하면? (BS 가격보다 클까 작을까)
    "wrong_price_mu015": None,
}

## 4. Simulate — 시뮬레이션

먼저 작은 도우미를 정의한다. 이 노트북에서 "정확"이라 부르는 숫자는 모두 `black_scholes_call/put` 과 여기서 계산된다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def d1d2(S0, K, r, sigma, T):
    """블랙-숄즈의 d1, d2."""
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return d1, d1 - sigma * np.sqrt(T)

def pde_residual(tau, x, h=1e-2, k=1e-4):
    """u(τ,x) = BS 콜 가격의 유한차분 잔차 -u_τ + r x u_x + ½σ²x²u_xx - r u (정리 2). (잔차, u_x, u_xx) 반환."""
    u = lambda tt, xx: black_scholes_call(xx, K, r, sigma, tt)
    u_tau = (u(tau + k, x) - u(tau - k, x)) / (2 * k)
    u_x = (u(tau, x + h) - u(tau, x - h)) / (2 * h)
    u_xx = (u(tau, x + h) - 2 * u(tau, x) + u(tau, x - h)) / h**2
    return -u_tau + r * x * u_x + 0.5 * sigma**2 * x**2 * u_xx - r * u(tau, x), u_x, u_xx

def real_drift_expected_payoff(S0, K, r, mu, sigma, T):
    """드리프트 μ 의 GBM 에 대한 e^{-rT} E[(S_T-K)^+] (정리 1 의 계산을 μ 로 반복; 가격이 아님)."""
    d1p, d2p = d1d2(S0, K, mu, sigma, T)
    return np.exp(-r * T) * (S0 * np.exp(mu * T) * norm.cdf(d1p) - K * norm.cdf(d2p))

def running_mean_band(x):
    """누적 평균과 ±1.96·누적 SE (Fig 2 용). (n, mean, half_width) 반환."""
    n = np.arange(1, len(x) + 1)
    m, m2 = np.cumsum(x) / n, np.cumsum(x**2) / n
    se = np.sqrt(np.maximum(m2 - m**2, 0.0) / np.maximum(n - 1, 1))
    return n, m, 1.96 * se

### Step 1 — 위험중립 GBM 경로와 $S_T$ 의 로그정규 분포

`gbm_paths` 에 드리프트로 $\mu$ 가 아니라 **$r$** 을 넣는다. 이것이 "위험중립 세계"의 전부다. $S_T$ 의 히스토그램을 `scipy.stats.lognorm(s=σ√T, scale=S0 e^{(r-σ²/2)T})` 의 밀도와 겹쳐 본다.

In [ ]:
N_STEPS = 250
t, S = gbm_paths(S0, r, sigma, N_PATHS, N_STEPS, T, rng)   # 드리프트 = r (위험중립)
ST_paths = S[:, -1]
ln = lognorm(s=sigma * np.sqrt(T), scale=S0 * np.exp((r - 0.5 * sigma**2) * T))
xx = np.linspace(40, 200, 400)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
plot_paths(t, S, ax=ax1, n_show=10)
ax1.axhline(K, color="k", ls="--", lw=1, label="K")
ax1.set(xlabel="t", ylabel="S_t", title="Risk-neutral GBM paths (μ = r = 0.05, σ = 0.2)")
ax1.legend(loc="upper left")
plot_hist_vs_pdf(ST_paths, xx, ln.pdf(xx), ax=ax2, label_pdf="log-normal pdf")
ax2.axvline(K, color="k", ls="--", lw=1, label="K")
ax2.set(xlabel="S_T", ylabel="density", title="Histogram of S_T vs log-normal density")
ax2.legend()
plt.show()

In [ ]:
d1, d2 = d1d2(S0, K, r, sigma, T)
E_ST_mc = mc_estimate(ST_paths)
p_itm_mc = mc_proportion(ST_paths > K)
print(f"d1 = {d1:.4f}, d2 = {d2:.4f}")
print(f"E*[S_T]     MC {E_ST_mc}   exact S0 e^(rT) = {S0 * np.exp(r * T):.4f}")
print(f"P*(S_T > K) MC {p_itm_mc}   exact Φ(d2)     = {norm.cdf(d2):.4f}")

평균은 $S_0e^{rT}$ 로 위험중립 세계에서 주식은 "무위험 이자율만큼" 자란다; 내가격 확률은 $\Phi(d_2)$ — 정리 1 의 둘째 조각이 벌써 보인다.

### Step 2 — MC 가격 vs 블랙-숄즈 공식, 그리고 안티테틱 teaser

`mc_option_price` 는 $S_T$ 를 로그정규로 정확히 뽑아 $e^{-rT}\,\text{payoff}$ 의 평균과 SE 를 돌려준다. 안티테틱(antithetic) 변량은 $(Z,-Z)$ 쌍의 평균을 한 표본으로 쓴다. `mc_option_price(..., n, antithetic=True)` 는 정규 난수를 `n//2` 개만 뽑아 `n//2` 쌍을 만들므로, **같은 정규 난수 개수** ($N_{MC}$ 개 → $N_{MC}$ 쌍) 로 비교하려면 `2 * N_MC` 를 넘긴다. 그때 SE 가 얼마나 줄어드는지 보자 (C4 의 분산감소 예고).

In [ ]:
call_payoff = lambda s: np.maximum(s - K, 0.0)
put_payoff = lambda s: np.maximum(K - s, 0.0)
call = mc_option_price(call_payoff, S0, r, sigma, T, N_MC, rng)
call_anti = mc_option_price(call_payoff, S0, r, sigma, T, 2 * N_MC, rng, antithetic=True)   # 정규 난수 N_MC 개 = N_MC 쌍
put = mc_option_price(put_payoff, S0, r, sigma, T, N_MC, rng)
bs_call, bs_put = black_scholes_call(S0, K, r, sigma, T), black_scholes_put(S0, K, r, sigma, T)
parity_mc, parity_se = call.mean - put.mean, np.hypot(call.se, put.se)   # 독립 표본이라 SE 는 제곱합의 제곱근
print(f"call        MC {call}   BS {bs_call:.6f}")
print(f"call (anti) MC {call_anti}   SE ratio plain/anti = {call.se / call_anti.se:.2f} (same number of normal draws)")
print(f"put         MC {put}   BS {bs_put:.6f}")
print(f"parity C-P  MC {parity_mc:.4f} ± {parity_se:.4f}   exact S0 - K e^(-rT) = {S0 - K * np.exp(-r * T):.6f}")

Fig 2 는 표본 수를 늘려 가며 누적 평균이 공식 값으로 수렴하는 모습이다. 여기서 뽑는 공통 난수 `Z` 는 Step 3·5 에서도 다시 쓴다.

In [ ]:
disc = np.exp(-r * T)
Z = rng.standard_normal(N_MC)                                       # 공통 난수 (Step 3, 5 재사용)
ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)   # 위험중립 S_T
ST_minus = S0 * np.exp((r - 0.5 * sigma**2) * T - sigma * np.sqrt(T) * Z)   # 짝 -Z (같은 난수, 두 번째 payoff 계산)
n_run, m_plain, b_plain = running_mean_band(disc * call_payoff(ST))
n_anti, m_anti, b_anti = running_mean_band(0.5 * disc * (call_payoff(ST) + call_payoff(ST_minus)))   # n 개 난수 = n 쌍

fig, ax = plt.subplots()
ax.fill_between(n_run, m_plain - b_plain, m_plain + b_plain, alpha=0.2, color="C0")
ax.plot(n_run, m_plain, color="C0", lw=1, label="plain MC (95% band)")
ax.fill_between(n_anti, m_anti - b_anti, m_anti + b_anti, alpha=0.2, color="C1")
ax.plot(n_anti, m_anti, color="C1", lw=1, label="antithetic pairs (95% band)")
ax.axhline(bs_call, color="k", ls="--", lw=1, label=f"Black–Scholes {bs_call:.4f}")
ax.set(xscale="log", xlim=(100, N_MC), ylim=(bs_call - 1.5, bs_call + 1.5),
       xlabel="number of normal draws", ylabel="running estimate of call price",
       title="Running MC estimate of the call price: plain vs antithetic")
ax.legend(loc="upper right")
plt.show()

두 띠 모두 공식 값을 품고 $1/\sqrt n$ 으로 좁아지며, 같은 정규 난수 개수에서 안티테틱 띠는 폭이 **절반** 이다 ($(Z,-Z)$ 의 payoff 가 음의 상관을 가지기 때문; 위 출력의 SE 비 $\approx2$). payoff 계산 횟수 (안티테틱은 난수당 두 번) 를 같게 놓고 비교하면 비는 $2/\sqrt2\approx1.4$ 로, 그래도 이득이다.

### Step 3 — 유도의 두 조각과 pathwise 델타

정리 1 의 증명은 할인 payoff 를 $e^{-rT}S_T\mathbf 1_{\{S_T>K\}}$ 와 $e^{-rT}K\mathbf 1_{\{S_T>K\}}$ 로 나누었다. 각 조각을 따로 MC 로 재고, 첫 조각을 $S_0$ 로 나눈 pathwise 델타 추정량 (정리 3) 도 함께 잰다.

In [ ]:
itm = ST > K
piece_asset = mc_estimate(disc * ST * itm)         # → S0 Φ(d1)
piece_cash = mc_estimate(disc * K * itm)           # → K e^{-rT} Φ(d2)
delta_pw = mc_estimate(disc * ST / S0 * itm)       # → Φ(d1)
exact_asset, exact_cash = S0 * norm.cdf(d1), K * disc * norm.cdf(d2)
print(f"E*[e^-rT S_T 1{{S_T>K}}]  MC {piece_asset}   exact S0 Φ(d1)      = {exact_asset:.6f}")
print(f"E*[e^-rT K   1{{S_T>K}}]  MC {piece_cash}   exact K e^-rT Φ(d2) = {exact_cash:.6f}")
print(f"difference                  {piece_asset.mean - piece_cash.mean:.4f}   BS call = {bs_call:.6f}")
print(f"delta (pathwise)          MC {delta_pw}   exact Φ(d1)         = {norm.cdf(d1):.6f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.hist(disc * ST[itm], bins=60, density=False, alpha=0.6, color="C0",
         weights=np.full(itm.sum(), 1.0 / N_MC), label="e^{-rT} S_T on {S_T > K}")
ax1.axvline(exact_asset, color="k", ls="--", lw=1, label=f"mean S0 Φ(d1) = {exact_asset:.2f} (incl. atom at 0)")
ax1.axvline(K * disc, color="C3", ls=":", lw=1, label=f"K e^{{-rT}} = {K * disc:.2f}")
ax1.set(xlabel="e^{-rT} S_T", ylabel="probability mass",
        title=f"Asset piece (atom of mass {1 - p_itm_mc.mean:.3f} at 0 not shown)")
ax1.legend(loc="upper right")
ax2.bar([0, K * disc], [1 - itm.mean(), itm.mean()], width=6, alpha=0.6, color="C1", label="MC")
ax2.plot([0, K * disc], [1 - norm.cdf(d2), norm.cdf(d2)], "ko", label="1-Φ(d2), Φ(d2)")
ax2.axvline(exact_cash, color="k", ls="--", lw=1, label=f"mean K e^{{-rT}} Φ(d2) = {exact_cash:.2f}")
ax2.set(xlabel="e^{-rT} K · 1{S_T > K}", ylabel="probability", title="Cash piece: two-point distribution")
ax2.legend(loc="upper left")
plt.show()

콜 가격 10.45 는 두 큰 수 63.68 과 53.23 의 차이다 — 이 때문에 MC 의 SE 가 상대적으로 크다 (2.6 절 함정 3).

### Step 4 — 파인만-카츠 검증: PDE 잔차, $u(0,x)$ 곡선, 다른 만기

(a) 블랙-숄즈 공식을 $u(\tau,x)$ 로 놓고 중앙차분으로 $u_\tau,u_x,u_{xx}$ 를 구해 BS PDE 잔차를 다섯 점에서 계산한다 (결정론적 검사). 덤으로 $u_x$ 는 델타 $\Phi(d_1)$, $u_{xx}$ 는 감마 $\phi(d_1)/(S_0\sigma\sqrt T)$ 와 맞아야 한다.

In [ ]:
test_points = [(1.0, 100.0), (0.5, 80.0), (0.25, 110.0), (2.0, 120.0), (1.0, 60.0)]
residuals = np.array([pde_residual(tau, x)[0] for tau, x in test_points])
for (tau, x), res in zip(test_points, residuals):
    print(f"(τ={tau:4}, x={x:5}): residual = {res: .2e}")
_, ux_fd, uxx_fd = pde_residual(1.0, 100.0)
gamma_exact = norm.pdf(d1) / (S0 * sigma * np.sqrt(T))
print(f"u_x(1,100)  finite diff {ux_fd:.6f}   Φ(d1) = {norm.cdf(d1):.6f}")
print(f"u_xx(1,100) finite diff {uxx_fd:.6f}   gamma = {gamma_exact:.6f}")
assert np.all(np.abs(residuals) < 1e-4), "BS formula does not satisfy the BS PDE?!"

(b) $u(0,x)$ 곡선: 초기 주가 $x\in\{60,\dots,140\}$ 마다 **공통 난수** `Z_grid` 로 MC 가격을 구해 (곡선이 매끄럽게 나오는 이유) 공식 곡선·payoff 와 함께 그린다. (c) 만기 $T=0.5$ 에서도 MC 와 공식을 비교한다.

In [ ]:
x_grid = np.arange(60, 141, 10)
Z_grid = rng.standard_normal(N_GRID)
growth = np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z_grid)   # S_T / S_0 (공통 난수)
u_mc = [mc_estimate(disc * np.maximum(x * growth - K, 0.0)) for x in x_grid]
u_bs = np.array([black_scholes_call(x, K, r, sigma, T) for x in x_grid])
u_mc_at_100 = u_mc[list(x_grid).index(100)]
T_half = 0.5
Z_half = rng.standard_normal(N_MC)
ST_half = S0 * np.exp((r - 0.5 * sigma**2) * T_half + sigma * np.sqrt(T_half) * Z_half)
call_half = mc_estimate(np.exp(-r * T_half) * call_payoff(ST_half))
bs_half = black_scholes_call(S0, K, r, sigma, T_half)
for x, u, b in zip(x_grid, u_mc, u_bs):
    print(f"x = {x:3d}: u(0,x) MC {u.mean:8.4f} ± {u.se:.4f}   BS {b:8.4f}")
print(f"T = 0.5: call MC {call_half}   BS {bs_half:.6f}")

In [ ]:
xf = np.linspace(50, 150, 300)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xf, [black_scholes_call(x, K, r, sigma, T) for x in xf], "k-", lw=1.5, label="Black–Scholes u(0,x)")
ax.plot(xf, np.maximum(xf - K, 0), "C3--", lw=1, label="payoff (x-K)^+")
ax.errorbar(x_grid, [u.mean for u in u_mc], yerr=[2 * u.se for u in u_mc], fmt="o", color="C0",
            capsize=3, label="MC ± 2 SE (common random numbers)")
ax.set(xlabel="initial price x", ylabel="call price", title="u(0,x): Black–Scholes curve vs Monte Carlo")
ax.legend(loc="upper left")
ins = ax.inset_axes([0.06, 0.40, 0.34, 0.30])
ins.bar(range(len(test_points)), np.abs(residuals) * 1e7, color="C2")
ins.set_xticks(range(len(test_points)), [f"({tau:g},{x:g})" for tau, x in test_points], fontsize=6, rotation=30)
ins.set(title="|PDE residual| × 1e7 at (τ, x)", ylabel="× 1e-7")
ins.title.set_fontsize(8)
plt.show()

MC 점이 공식 곡선 위에 놓이고 (같은 $Z$ 를 써서 오차가 $x$ 에 따라 매끄럽게 변한다), 유한차분 잔차는 $10^{-7}$ 수준이다: 위험중립 기댓값과 PDE 의 해가 같은 함수다 (정리 2).

### Step 5 — 왜 $\mu$ 가 사라지는가: 실제 드리프트로 계산해 보기

가정을 깨 보자. 드리프트를 실제 값 $\mu=0.15$ 로 둔 GBM 으로 같은 할인 기댓값을 계산하면 (정리 1 의 계산을 $\mu$ 로 반복한 `real_drift_expected_payoff` 가 닫힌 형식) BS 가격과 전혀 다른 수가 나온다. 이것은 '가격'이 아니라 '할인 기대보수'다.

In [ ]:
mu_real = 0.15
ST_real = S0 * np.exp((mu_real - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)   # 같은 Z, 드리프트만 μ
wrong_mc = mc_estimate(disc * call_payoff(ST_real))
wrong_exact = real_drift_expected_payoff(S0, K, r, mu_real, sigma, T)
p_itm_real = mc_proportion(ST_real > K)
_, d2_real = d1d2(S0, K, mu_real, sigma, T)
print(f"e^-rT E[(S_T-K)^+] with μ = {mu_real}:  MC {wrong_mc}   closed form {wrong_exact:.6f}   (BS price {bs_call:.4f})")
print(f"real P(S_T > K):  MC {p_itm_real}   Φ(d2') = {norm.cdf(d2_real):.6f}   (risk-neutral Φ(d2) = {norm.cdf(d2):.6f})")

In [ ]:
mu_grid = np.linspace(-0.05, 0.25, 13)
vals = [mc_estimate(disc * call_payoff(S0 * np.exp((m - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z))) for m in mu_grid]
mu_fine = np.linspace(-0.05, 0.25, 200)
fig, ax = plt.subplots()
ax.plot(mu_fine, [real_drift_expected_payoff(S0, K, r, m, sigma, T) for m in mu_fine], "k-", lw=1, label="closed form")
ax.errorbar(mu_grid, [v.mean for v in vals], yerr=[2 * v.se for v in vals], fmt="o", color="C0", capsize=3, label="MC ± 2 SE")
ax.plot([r], [bs_call], "*", color="C3", ms=14, label=f"μ = r: Black–Scholes price {bs_call:.4f}")
ax.axvline(r, color="C3", ls=":", lw=1)
ax.set(xlabel="drift μ used in the simulation", ylabel="e^{-rT} E[(S_T - K)^+]",
       title="Discounted expected payoff depends on μ; the price is the μ = r point")
ax.legend(loc="upper left")
plt.show()

할인 기대보수는 $\mu$ 에 따라 크게 변한다. 그중 **하나의 점** ($\mu=r$) 만이 복제 비용, 즉 무차익 가격이다. $\mu$ 가 다른 두 투자자는 이 옵션의 '기대 이익'에는 동의하지 않지만 헤지 비용에는 동의한다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
exact = {
    "bs_call": bs_call, "bs_call_anti": bs_call, "bs_put": bs_put,
    "E_ST": S0 * np.exp(r * T), "p_itm": norm.cdf(d2),
    "piece_asset": exact_asset, "piece_cash": exact_cash, "delta": norm.cdf(d1),
    "u_100": bs_call, "call_half": bs_half, "wrong_price_mu015": wrong_exact,
    "parity": S0 - K * np.exp(-r * T),
}
rows = [
    {"name": "call price", "predicted": predictions.get("bs_call"), "estimate": call, "exact": exact["bs_call"]},
    {"name": "call price (antithetic)", "predicted": None, "estimate": call_anti, "exact": exact["bs_call_anti"]},
    {"name": "put price", "predicted": predictions.get("bs_put"), "estimate": put, "exact": exact["bs_put"]},
    {"name": "put-call parity C − P (table only)", "predicted": None, "estimate": parity_mc, "se": parity_se, "exact": exact["parity"]},
    {"name": "E*[S_T]", "predicted": predictions.get("E_ST"), "estimate": E_ST_mc, "exact": exact["E_ST"]},
    {"name": "P*(S_T > K)", "predicted": predictions.get("p_itm"), "estimate": p_itm_mc, "exact": exact["p_itm"]},
    {"name": "E*[e^-rT S_T 1{S_T>K}] = S0 Φ(d1)", "predicted": None, "estimate": piece_asset, "exact": exact["piece_asset"]},
    {"name": "E*[e^-rT K 1{S_T>K}] = K e^-rT Φ(d2)", "predicted": None, "estimate": piece_cash, "exact": exact["piece_cash"]},
    {"name": "delta (pathwise)", "predicted": predictions.get("delta"), "estimate": delta_pw, "exact": exact["delta"]},
    {"name": "u(0,100) from the x-grid MC", "predicted": None, "estimate": u_mc_at_100, "exact": exact["u_100"]},
    {"name": "call, T = 0.5", "predicted": None, "estimate": call_half, "exact": exact["call_half"]},
    {"name": "discounted expected payoff, real drift μ=0.15 (NOT a price)", "predicted": predictions.get("wrong_price_mu015"), "estimate": wrong_mc, "exact": exact["wrong_price_mu015"]},
    {"name": "BS PDE residual, max abs value × 1e7 (deterministic; tolerance 1e-4 → 1000)", "predicted": None, "estimate": float(np.max(np.abs(residuals)) * 1e7), "exact": None},
]
print(f"max |PDE residual| = {np.max(np.abs(residuals)):.1e} (deterministic, tolerance 1e-4; asserted below)")
Markdown(compare_table(rows))

각 행에서 불일치가 뜻하는 것:
- **call / put / call (antithetic)**: 공식과 4 SE 밖이면 `mc_option_price` 의 $S_T$ 표본 (특히 $-\sigma^2/2$ 보정) 이나 할인이 잘못된 것. 안티테틱 행은 값이 같되 SE 가 절반 ($\approx0.016$ vs $0.033$, 같은 정규 난수 개수) 이어야 한다.
- **put-call parity**: 두 독립 MC 의 차이라 SE 는 $\sqrt{se_C^2+se_P^2}$; 공식 자체의 패리티는 아래에서 $10^{-10}$ 로 따로 확인한다.
- **E*[S_T]**: $S_0e^{rT}$ 와 다르면 `gbm_paths` 에 $r$ 이 아닌 드리프트를 넣었거나 브라운 증분 분산이 틀린 것.
- **P*(S_T > K)**: $\Phi(d_2)$ 와 다르면 $d_2$ 의 부호 ($r-\sigma^2/2$, $+\sigma^2/2$ 아님) 를 의심.
- **두 조각**: 정리 1 증명의 완전제곱 단계가 틀렸다면 첫 조각이 $S_0\Phi(d_2)$ 나 $S_0e^{rT}\Phi(d_1)$ 근처로 나온다.
- **delta (pathwise)**: $\Phi(d_1)$ 과 다르면 $\partial S_T/\partial S_0=S_T/S_0$ 대신 1 을 쓴 것 — 그러면 $e^{-rT}\Phi(d_2)$ 가 나온다.
- **u(0,100)**, **call T=0.5**: 파인만-카츠의 MC 쪽 — 공통 난수 곡선이 공식 곡선에서 계통적으로 벗어나면 `growth` 의 $T$ 와 할인의 $T$ 가 어긋난 것.
- **real drift μ=0.15**: 닫힌 형식 18.08 과 맞아야 한다. 이 행이 BS 행과 **다른 것이 정상**이다 — 같으면 드리프트를 바꾸지 않은 것.
- **PDE residual**: 결정론적 값이라 CI·판정 없이 $10^7$ 배로 표시했다 (마지막 셀의 assert 가 실제 검사). $10^{-4}$ 를 넘으면 $u_t=-u_\tau$ 부호나 $\frac12\sigma^2x^2$ 계수를 의심 (2.6 절 함정 4).

In [ ]:
assert_close(call, exact["bs_call"], k=4, name="MC call")
assert_close(call_anti, exact["bs_call_anti"], k=4, name="MC call antithetic")
assert_close(put, exact["bs_put"], k=4, name="MC put")
assert_close(E_ST_mc, exact["E_ST"], k=4, name="E*[S_T]")
assert_close(p_itm_mc, exact["p_itm"], k=4, name="P*(S_T>K)")
assert_close(piece_asset, exact["piece_asset"], k=4, name="piece S0 Phi(d1)")
assert_close(piece_cash, exact["piece_cash"], k=4, name="piece K e^-rT Phi(d2)")
assert_close(delta_pw, exact["delta"], k=4, name="delta pathwise")
assert_close(u_mc_at_100, exact["u_100"], k=4, name="u(0,100)")
assert_close(call_half, exact["call_half"], k=4, name="call T=0.5")
assert_close(wrong_mc, exact["wrong_price_mu015"], k=4, name="real-drift expected payoff (closed form)")
assert_close(p_itm_real, norm.cdf(d2_real), k=4, name="real P(S_T>K)")
# 결정론적 검사: 파인만-카츠 잔차와 공식의 풋-콜 패리티
assert np.all(np.abs(residuals) < 1e-4), "Feynman-Kac finite-difference check failed"
assert abs(bs_call - bs_put - exact["parity"]) < 1e-10, "put-call parity of the formulas failed"
print(f"all assert_close passed (k=4); SE ratio plain/antithetic = {call.se / call_anti.se:.2f}; max |PDE residual| = {np.max(np.abs(residuals)):.1e}")

## 6. 연습문제

### E1 계산

$S_0=100,K=100,r=0.05,\sigma=0.2,T=1$ 에서 $d_1,d_2$ 와 $\Phi$ 표로 콜·풋 가격을 **손으로** 구하고 패리티를 확인하라. 또 실제 드리프트 $\mu=0.15$ 일 때 실제 행사확률 $P(S_T>K)$ 를 구해 위험중립 확률 $\Phi(d_2)$ 와 비교하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$d_1=\dfrac{\log 1+(0.05+0.02)\cdot1}{0.2}=0.35$, $d_2=0.35-0.2=0.15$. $\Phi(0.35)=0.6368$, $\Phi(0.15)=0.5596$, $Ke^{-rT}=95.1229$.
$C=100\times0.6368-95.1229\times0.5596=63.68-53.23=10.45$. 패리티로 $P=C-S_0+Ke^{-rT}=10.45-100+95.12=5.57$ (직접: $95.1229\times0.4404-100\times0.3632=5.57$).

실제 드리프트: $d_2'=\dfrac{(0.15-0.02)\cdot1}{0.2}=0.65$, $P(S_T>K)=\Phi(0.65)=0.7422$ — 위험중립 $0.5596$ 보다 훨씬 크지만 가격에는 들어가지 않는다.

```python
d1, d2 = d1d2(100, 100, 0.05, 0.2, 1.0)
C = 100 * norm.cdf(d1) - 100 * np.exp(-0.05) * norm.cdf(d2)      # 10.450584
P = C - 100 + 100 * np.exp(-0.05)                                 # 5.573526
p_real = norm.cdf(d1d2(100, 100, 0.15, 0.2, 1.0)[1])              # 0.742154
```

</details>

### E2 유도 후 시뮬레이션 검증

디지털(cash-or-nothing) 콜 $\mathbf 1_{\{S_T>K\}}$ 의 가격 $e^{-rT}\Phi(d_2)$ 와 asset-or-nothing 콜 $S_T\mathbf 1_{\{S_T>K\}}$ 의 가격 $S_0\Phi(d_1)$ 을 유도하고 (정리 1 의 두 조각), MC 로 검증하라. 유럽형 콜 $=$ asset-or-nothing $-\,K\times$ cash-or-nothing 임을 확인하라. 디지털 콜의 델타를 pathwise 로 추정하면 왜 0 이 나오는지 설명하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** cash-or-nothing: $E^*[e^{-rT}\mathbf 1_{\{S_T>K\}}]=e^{-rT}P^*(Z>-d_2)=e^{-rT}\Phi(d_2)=e^{-0.05}\times0.559618=0.532325$. asset-or-nothing: 정리 1 첫 조각 그대로 $S_0\Phi(d_1)=63.683065$. 콜 $=(S_T-K)\mathbf 1_{\{S_T>K\}}=S_T\mathbf 1_{\{S_T>K\}}-K\mathbf 1_{\{S_T>K\}}$ 이므로 $63.683-100\times0.532325=10.4506$.

**pathwise 델타가 0 인 이유.** 디지털 payoff $g(S_T)=\mathbf 1_{\{S_T>K\}}$ 는 계단함수라 $g'(S_T)=0$ a.s. ($S_T=K$ 는 확률 0). 그러므로 $E[e^{-rT}g'(S_T)S_T/S_0]=0$. 그러나 참 델타는 $\partial_{S_0}[e^{-rT}\Phi(d_2)]=e^{-rT}\phi(d_2)/(S_0\sigma\sqrt T)=0.01876\ne0$ (여기서는 $K=S_0$ 이고 $S_0\phi(d_1)=Ke^{-rT}\phi(d_2)$ 라 콜의 감마와 같은 수다). 미분과 기댓값의 교환이 깨진 것 (정리 3 의 Lipschitz 가정 위반); 밀도를 미분하는 likelihood-ratio 법이 필요하다 (C4). 아래 코드의 범프(bump) 추정량은 이를 눈으로 보여 준다: $S_0$ 를 $(1+\varepsilon)$ 배 하면 $S_T$ 도 $(1+\varepsilon)$ 배 되므로 유한차분 $[g(S_T(1+\varepsilon))-g(S_T)]/(\varepsilon S_0)$ 은 $S_T\in(K/(1+\varepsilon),K]$ 인 표본에서만 (크기 $1/(\varepsilon S_0)$ 로) 0 이 아니다. 그 집합의 확률은 $\propto\varepsilon$ 이라 기댓값은 참 델타로 가지만, $\varepsilon\to0$ 의 pathwise 극한에서는 걸리는 표본이 사라져 추정량이 정확히 0 이 된다 — 극한과 기댓값의 순서가 바뀌지 않는다.

```python
Z2 = rng.standard_normal(N_MC)
ST2 = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z2)
cash = mc_estimate(disc * (ST2 > K))            # ≈ 0.5323 (exact e^-rT Φ(d2) = 0.532325)
asset = mc_estimate(disc * ST2 * (ST2 > K))     # ≈ 63.68  (exact S0 Φ(d1) = 63.683065)
assert_close(cash, disc * norm.cdf(d2), k=4, name="cash-or-nothing")
assert_close(asset, S0 * norm.cdf(d1), k=4, name="asset-or-nothing")
print(asset.mean - K * cash.mean, bs_call)      # 두 조각의 차이 = 콜 가격
digital_delta_true = disc * norm.pdf(d2) / (S0 * sigma * np.sqrt(T))  # 0.018762 (= 콜의 감마, K = S0 이므로)
g = lambda s: (s > K).astype(float)
for eps in [1e-2, 1e-4, 1e-6, 1e-8]:                                   # 범프 추정량: ε → 0 이 pathwise 극한
    bump = mc_estimate(disc * (g(ST2 * (1 + eps)) - g(ST2)) / (eps * S0))
    n_cross = int(((ST2 <= K) & (ST2 * (1 + eps) > K)).sum())          # K 를 건너는 표본 수 ∝ ε
    print(f"eps={eps:.0e}: crossings {n_cross:5d}, bump delta {bump}   (true {digital_delta_true:.4f})")
# eps=1e-2: ≈0.019 ± 0.0003 (거의 참값) → eps=1e-8: crossings 0, 추정 0 — 극한에서 pathwise 미분은 a.s. 0
```

</details>

### E3 가정을 깨보기

상수 변동성 가정을 깨자. $\sigma$ 가 만기까지 $0.1$ 또는 $0.3$ (각 확률 $1/2$, $Z$ 와 독립) 인 **혼합 모형**에서 $K=120$ 콜의 할인 기대보수를 MC 로 구하고, $\sigma$ 로 조건화하여 얻는 닫힌 형식 $\frac12[BS(\sigma=0.1)+BS(\sigma=0.3)]$ 과 비교하라. $BS(\sigma=0.2)$ 와 왜 다른가? 이 값을 재현하는 내재변동성(implied volatility)을 `scipy.optimize.brentq` 로 구하라 (변동성 스마일 teaser).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$\sigma$ 로 조건화하면 각 가지는 상수 변동성 BS 세계이므로 $E[e^{-rT}(S_T-K)^+]=\frac12BS(0.1)+\frac12BS(0.3)=\frac12(0.4625+6.9040)=3.6832$. 반면 평균 변동성으로 계산한 $BS(0.2)=3.2475$. 콜 가격은 $\sigma$ 에 대해 (OTM 에서) **볼록**하므로 젠센 부등식으로 혼합 가격 $>BS(\bar\sigma)$. 이 가격을 재현하는 내재변동성은 $0.2126>0.2$: 외가격 행사가에서 내재변동성이 높아지는 **스마일**의 원형이다. (ATM $K=100$ 에서는 $10.518$ vs $10.451$ 로 거의 선형이라 차이가 작고 내재변동성 $0.2018$.) 정리 1 의 "로그정규 $S_T$" 가정이 무너지면 공식은 조각별 평균으로만 살아남는다.

```python
K3 = 120.0
sig_mix = rng.choice([0.1, 0.3], size=N_MC)                    # Z 와 독립인 σ
Z3 = rng.standard_normal(N_MC)
ST_mix = S0 * np.exp((r - 0.5 * sig_mix**2) * T + sig_mix * np.sqrt(T) * Z3)
mix_mc = mc_estimate(disc * np.maximum(ST_mix - K3, 0.0))
mix_exact = 0.5 * (black_scholes_call(S0, K3, r, 0.1, T) + black_scholes_call(S0, K3, r, 0.3, T))   # 3.6832
bs_avg_sigma = black_scholes_call(S0, K3, r, 0.2, T)                                                 # 3.2475
assert_close(mix_mc, mix_exact, k=4, name="mixture price")
iv = brentq(lambda s: black_scholes_call(S0, K3, r, s, T) - mix_exact, 1e-3, 2.0)                    # 0.2126
print(mix_mc, mix_exact, bs_avg_sigma, iv)
```

</details>

## 7. 정리

1. **GBM 해** $S_T=S_0e^{(\mu-\sigma^2/2)T+\sigma B_T}$ ($\log S$ 에 Itô); 가격은 드리프트를 $r$ 로 바꾼 위험중립 세계의 할인 기댓값 $E^*[e^{-rT}(S_T-K)^+]$.
2. **블랙-숄즈** $C=S_0\Phi(d_1)-Ke^{-rT}\Phi(d_2)$ 는 로그정규 적분을 두 조각으로 닫은 것: $\Phi(d_2)$ = 위험중립 행사확률, $S_0\Phi(d_1)$ = 행사 시 받는 주식의 현재가치. 둘 다 MC 로 4 SE 이내 확인.
3. **MC 가격의 SE** $\approx0.033$ (200 000 경로); 안티테틱은 같은 정규 난수 개수에서 SE 를 절반으로 줄인다 (payoff 계산 횟수 기준으로는 $\approx1.4$ 배) — C4 분산감소의 예고.
4. **파인만-카츠**: $u(t,x)=E^*[e^{-r(T-t)}f(S_T)\mid S_t=x]$ 는 BS PDE 의 해 ($e^{-rt}u(t,S_t)$ 에 Itô, 드리프트 0 → 마팅게일); 공식의 유한차분 잔차 $\sim10^{-7}$, $u(0,x)$ 곡선이 MC 와 일치, $u_x=\Phi(d_1)$ (pathwise 델타).
5. 실제 드리프트 $\mu=0.15$ 로 계산한 할인 기대보수 (18.08) 는 가격 (10.45) 이 아니다 — 가격은 복제 비용이며 $\mu$ 에 무관.

**Trap 카드.** Q: 옵션 가격은 주가의 기대 상승률 $\mu$ 에 의존하는가?
→ A: 아니다. 가격은 복제 포트폴리오의 비용이며 위험중립 측도 (드리프트 $r$) 의 할인 기댓값이다. $\mu=0.15$ 를 넣으면 18.08 이 나오지만 그것은 '기대 보수'이지 무차익 가격 (10.45) 이 아니다.

**다음 노트북: C4 (캡스톤 4 — 옵션 가격 결정).** 닫힌 형식이 없는 아시안·배리어 옵션을 07c 의 EM/밀스타인 경로와 오늘의 위험중립 프레임으로 MC 가격 매기고, 00b 의 MC 오차 이론으로 안티테틱·제어변량 분산감소를 설계한다. 오늘의 pathwise 델타 추정은 Greeks 절의 출발점이다.

`log/progress.md` 한 줄 템플릿:
`- [ ] 07e GBM·블랙-숄즈·파인만-카츠 — 날짜: ____ / 소요: ___분 / 예측 적중: _/6 / 메모: ____`